# 02 - LoRA Fine-Tune SmolVLM

This notebook fine-tunes `HuggingFaceTB/SmolVLM-256M-Instruct` on the synthetic glyph VQA dataset using LoRA adapters.

The base model stays frozen. LoRA trains small adapter matrices and saves them to Google Drive.

## Runtime

Use a GPU runtime. Colab Pro with L4, A100, or V100 is plenty for this 256M model. T4 should also work, though it may be slower.

In [ ]:
# Run this once at the top of a fresh Colab runtime.
%pip -q install -U transformers datasets accelerate peft trl bitsandbytes pillow matplotlib pandas

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/smolvlm_lora_lab")
except Exception:
    PROJECT_DIR = Path.cwd() / "smolvlm_lora_lab"

DATA_DIR = PROJECT_DIR / "synthetic_glyph_vqa"
IMAGE_DIR = DATA_DIR / "images"
ADAPTER_DIR = PROJECT_DIR / "smolvlm_256m_glyph_lora_adapter"

for path in [PROJECT_DIR, DATA_DIR, IMAGE_DIR, ADAPTER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Dataset dir:", DATA_DIR)
print("Adapter dir:", ADAPTER_DIR)

In [ ]:
import json
import random
from pathlib import Path

import pandas as pd
from PIL import Image, ImageDraw

RNG_SEED = 42
NUM_SCENES = 48

COLORS = {
    "teal": (20, 184, 166),
    "magenta": (217, 70, 239),
    "amber": (245, 158, 11),
    "lime": (132, 204, 22),
    "blue": (59, 130, 246),
    "rose": (244, 63, 94),
}

SHAPES = ["circle", "square", "triangle", "diamond"]
COUNTS = [1, 2, 3, 4]
DIRECTIONS = ["up", "right", "down", "left"]

COLOR_CODE = {
    "teal": "luma",
    "magenta": "voro",
    "amber": "kivo",
    "lime": "nexo",
    "blue": "safi",
    "rose": "pavo",
}

SHAPE_CODE = {
    "circle": "orbit",
    "square": "block",
    "triangle": "spire",
    "diamond": "facet",
}

ACTION_CODE = {
    "up": "mip-forward",
    "right": "mip-right",
    "down": "mip-back",
    "left": "mip-left",
}


def draw_arrow(draw, direction, box):
    x0, y0, x1, y1 = box
    cx = (x0 + x1) // 2
    cy = (y0 + y1) // 2
    if direction == "up":
        draw.line((cx, y1, cx, y0 + 10), fill=(20, 20, 20), width=7)
        draw.polygon([(cx, y0), (cx - 13, y0 + 18), (cx + 13, y0 + 18)], fill=(20, 20, 20))
    elif direction == "down":
        draw.line((cx, y0, cx, y1 - 10), fill=(20, 20, 20), width=7)
        draw.polygon([(cx, y1), (cx - 13, y1 - 18), (cx + 13, y1 - 18)], fill=(20, 20, 20))
    elif direction == "right":
        draw.line((x0, cy, x1 - 10, cy), fill=(20, 20, 20), width=7)
        draw.polygon([(x1, cy), (x1 - 18, cy - 13), (x1 - 18, cy + 13)], fill=(20, 20, 20))
    else:
        draw.line((x1, cy, x0 + 10, cy), fill=(20, 20, 20), width=7)
        draw.polygon([(x0, cy), (x0 + 18, cy - 13), (x0 + 18, cy + 13)], fill=(20, 20, 20))


def make_scene_image(scene, size=384):
    image = Image.new("RGB", (size, size), (248, 248, 244))
    draw = ImageDraw.Draw(image)
    draw.rounded_rectangle((18, 18, size - 18, size - 18), radius=24, outline=(35, 35, 35), width=4)
    draw.rectangle((34, 34, size - 34, size - 34), outline=(214, 214, 206), width=2)

    color = COLORS[scene["color"]]
    cx, cy = size // 2, size // 2 + 8
    r = 76
    if scene["shape"] == "circle":
        draw.ellipse((cx - r, cy - r, cx + r, cy + r), fill=color, outline=(20, 20, 20), width=4)
    elif scene["shape"] == "square":
        draw.rectangle((cx - r, cy - r, cx + r, cy + r), fill=color, outline=(20, 20, 20), width=4)
    elif scene["shape"] == "triangle":
        draw.polygon([(cx, cy - 92), (cx - 92, cy + 72), (cx + 92, cy + 72)], fill=color, outline=(20, 20, 20))
        draw.line([(cx, cy - 92), (cx - 92, cy + 72), (cx + 92, cy + 72), (cx, cy - 92)], fill=(20, 20, 20), width=4)
    else:
        draw.polygon([(cx, cy - 96), (cx - 96, cy), (cx, cy + 96), (cx + 96, cy)], fill=color, outline=(20, 20, 20))
        draw.line([(cx, cy - 96), (cx - 96, cy), (cx, cy + 96), (cx + 96, cy), (cx, cy - 96)], fill=(20, 20, 20), width=4)

    dot_y = size - 72
    start_x = cx - ((scene["count"] - 1) * 28) // 2
    for i in range(scene["count"]):
        x = start_x + i * 28
        draw.ellipse((x - 9, dot_y - 9, x + 9, dot_y + 9), fill=(20, 20, 20))

    draw_arrow(draw, scene["direction"], (size - 96, 46, size - 42, 100))
    return image


def glyph_answer(scene):
    return f"{COLOR_CODE[scene['color']]}-{SHAPE_CODE[scene['shape']]}-{scene['count']}"


QUESTION_SPECS = [
    (
        "glyph_code",
        "What is the synthetic glyph code for this panel? Answer only the code.",
        glyph_answer,
    ),
    (
        "dot_count",
        "How many black marker dots are shown? Answer with one number.",
        lambda scene: str(scene["count"]),
    ),
    (
        "arrow_direction",
        "What direction does the black arrow point? Answer one word.",
        lambda scene: scene["direction"],
    ),
    (
        "control_token",
        "What control token belongs to the arrow direction? Answer only the token.",
        lambda scene: ACTION_CODE[scene["direction"]],
    ),
]


def create_dataset(force=False):
    metadata_path = DATA_DIR / "metadata.jsonl"
    if metadata_path.exists() and not force:
        rows = [json.loads(line) for line in metadata_path.read_text().splitlines()]
        return pd.DataFrame(rows)

    random.seed(RNG_SEED)
    combos = [
        {"color": c, "shape": s, "count": n, "direction": d}
        for c in COLORS
        for s in SHAPES
        for n in COUNTS
        for d in DIRECTIONS
    ]
    random.shuffle(combos)
    scenes = combos[:NUM_SCENES]

    rows = []
    for scene_id, scene in enumerate(scenes):
        image = make_scene_image(scene)
        file_name = f"scene_{scene_id:03d}.png"
        image.save(IMAGE_DIR / file_name)

        split = "train" if scene_id < int(NUM_SCENES * 0.8) else "eval"
        for question_kind, question, answer_fn in QUESTION_SPECS:
            rows.append(
                {
                    "scene_id": scene_id,
                    "split": split,
                    "file_name": file_name,
                    "color": scene["color"],
                    "shape": scene["shape"],
                    "count": scene["count"],
                    "direction": scene["direction"],
                    "question_kind": question_kind,
                    "question": question,
                    "answer": answer_fn(scene),
                }
            )

    with metadata_path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    return pd.DataFrame(rows)


df = create_dataset(force=False)
print(df.shape)
display(df.head())
print(df.groupby(["split", "question_kind"]).size())

In [ ]:
from PIL import Image

SYSTEM_MESSAGE = (
    "You are answering questions about synthetic glyph panels. "
    "Give the exact short answer requested. Do not explain."
)


def row_to_vlm_sample(row):
    image = Image.open(IMAGE_DIR / row["file_name"]).convert("RGB")
    return {
        "images": [image],
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_MESSAGE}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": row["question"]},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": row["answer"]}],
            },
        ],
    }


train_rows = df[df["split"] == "train"].reset_index(drop=True)
eval_rows = df[df["split"] == "eval"].reset_index(drop=True)
train_dataset = [row_to_vlm_sample(row) for _, row in train_rows.iterrows()]
eval_dataset = [row_to_vlm_sample(row) for _, row in eval_rows.iterrows()]

print("Train samples:", len(train_dataset))
print("Eval samples:", len(eval_dataset))
print(train_dataset[0]["messages"][1]["content"][1]["text"])
print("Answer:", train_dataset[0]["messages"][2]["content"][0]["text"])

## Fine-tuning setup

This follows the same basic pattern as the Hugging Face SmolVLM fine-tuning cookbook:

1. Load a VLM and processor.
2. Format each example as messages plus images.
3. Configure LoRA.
4. Train with TRL `SFTTrainer`.
5. Save the adapter.

Important VLM detail: `max_length=None` keeps training from accidentally truncating image tokens.

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
USE_4BIT = False  # Keep False for the most predictable Colab Pro path. Set True if VRAM is tight.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("This training notebook expects a GPU runtime. In Colab: Runtime -> Change runtime type -> GPU.")

major, _ = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("GPU:", torch.cuda.get_device_name(0))
print("bf16:", USE_BF16)
print("adapter output:", ADAPTER_DIR)

model_kwargs = {
    "torch_dtype": TORCH_DTYPE,
    "device_map": "auto",
    "_attn_implementation": "eager",
}

if USE_4BIT:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=TORCH_DTYPE,
    )

model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, **model_kwargs)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.config.use_cache = False

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = SFTConfig(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    optim="adamw_torch",
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    max_length=None,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=processor,
)

trainer.train()
trainer.save_model(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))

print("Saved LoRA adapter to:", ADAPTER_DIR)

In [ ]:
from pathlib import Path

expected_files = ["adapter_config.json", "adapter_model.safetensors"]
for name in expected_files:
    path = ADAPTER_DIR / name
    print(name, "OK" if path.exists() else "MISSING", path)

## What got saved

The important files are:

- `adapter_config.json`
- `adapter_model.safetensors`

Those are the LoRA adapter. You still need the base SmolVLM model when you reload the adapter.